In [1]:
%pip install -r requirements-offline.txt --quiet

/Users/utarn/projects/ml-metrology/.claude/worktrees/issue-26/.check-env/bin/python: No module named pip


Note: you may need to restart the kernel to use updated packages.


# Exercise — Module M1: เทรน 3 แบบบนข้อมูลเดียวกัน

**วิธีทำ:** เติมช่อง `# TODO` / `...` ให้ทุกเซลล์รันผ่าน แล้วตอบคำถาม markdown ที่กำหนด — เปิด `book.ipynb` เทียบได้ตลอด (โค้ดชุดเดียวกัน)

**Dataset:** Fashion-MNIST — Xiao et al. (2017), arXiv:1708.07747, <https://github.com/zalandoresearch/fashion-mnist>, **License: MIT**

**เวลาโดยประมาณ:** 10–15 นาที (เทรนทั้งหมดสั้นโดยตั้งใจ)

## 0. เตรียมข้อมูล (ให้มาแล้ว — รันผ่าน)

In [2]:
from pathlib import Path


def repo_root() -> Path:
    """หา repo root โดยไล่ขึ้นจาก cwd."""
    for p in (Path.cwd(), *Path.cwd().parents):
        if (p / "requirements-offline.txt").exists():
            return p
    raise SystemExit(
        "ไม่พบ repo root — เปิด JupyterLab จาก repo root (uv run jupyter lab)"
    )


DATA = repo_root() / "data"

In [3]:
import numpy as np  # noqa: E402
from torchvision.datasets import FashionMNIST  # noqa: E402

# download=True: ดาวน์โหลดครั้งแรกลง data/ (gitignored) — บนเครื่อง
# offline ที่ pre-bundle ไว้แล้ว จะโหลดจากดิสก์ทันทีไม่ใช้เน็ต
train_full = FashionMNIST(root=DATA, train=True, download=True)
test_full = FashionMNIST(root=DATA, train=False, download=True)
print(train_full.data.shape, test_full.data.shape)

torch.Size([60000, 28, 28]) torch.Size([10000, 28, 28])


In [4]:
# fixed subsample 10k/2k — เพื่อให้ทั้ง lab จบในเวลาไม่กี่นาทีบน CPU
# (ผลเมื่อใช้เต็ม 60k/10k อ้างอิงไว้ในสรุปท้ายโมดูล)
N_TRAIN, N_TEST = 10_000, 2_000
rng = np.random.default_rng(42)
idx_tr = rng.choice(len(train_full.data), N_TRAIN, replace=False)
idx_te = rng.choice(len(test_full.data), N_TEST, replace=False)
X_train = train_full.data.numpy()[idx_tr]
y_train = train_full.targets.numpy()[idx_tr]
X_test = test_full.data.numpy()[idx_te]
y_test = test_full.targets.numpy()[idx_te]
print(X_train.shape, X_test.shape)

(10000, 28, 28) (2000, 28, 28)


In [5]:
CLASS_EN = [
    "T-shirt", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot",
]

## ข้อ 1 — HOG: เล่นกับขนาด cell

ฟังก์ชัน `hog_features` รับ `cell` (ขนาด cell เป็นพิกเซล) — book.ipynb ใช้ `cell=4` ได้ accuracy ~0.866 จงเทรน HOG+SVM ด้วย **`cell=7`** (coarser — ได้ 144 มิติ) แล้วเทียบ accuracy

**คำถาม:** cell ใหญ่ขึ้น accuracy เปลี่ยนไปทางไหน และเพราะอะไร (ตอบใน markdown)

In [6]:
import numpy as np  # noqa: E402


def hog_features(imgs, cell=4, bins=9):
    """HOG แบบย่อ: gradient -> histogram มุมต่อ cell -> L2 normalize.

    imgs: (N, 28, 28) uint8 -> return (N, (28/cell)^2 * bins) float32
    """
    imgs = imgs.astype(np.float32)
    gx = np.empty_like(imgs)
    gy = np.empty_like(imgs)
    gx[:, :, 1:-1] = imgs[:, :, 2:] - imgs[:, :, :-2]
    gy[:, 1:-1, :] = imgs[:, 2:, :] - imgs[:, :-2, :]
    mag = np.hypot(gx, gy)
    ang = np.arctan2(gy, gx) % np.pi
    ny, nx = imgs.shape[1] // cell, imgs.shape[2] // cell
    b = np.floor(ang / (np.pi / bins)).astype(int) % bins
    feats = np.zeros((len(imgs), ny * nx * bins), dtype=np.float32)
    for i in range(len(imgs)):
        f = feats[i].reshape(ny, nx, bins)
        for cy in range(ny):
            for cx in range(nx):
                sl_m = mag[i, cy * cell:(cy + 1) * cell,
                           cx * cell:(cx + 1) * cell]
                sl_b = b[i, cy * cell:(cy + 1) * cell,
                         cx * cell:(cx + 1) * cell]
                f[cy, cx] = np.bincount(
                    sl_b.ravel(), weights=sl_m.ravel(), minlength=bins
                )
        feats[i] = (f / (np.linalg.norm(f) + 1e-6)).ravel()
    return feats

In [7]:
# คำนวณ HOG feature ด้วย cell=4 (ค่า default ของ book.ipynb) — ให้มาแล้ว
Ftr_hog = hog_features(X_train)
Fte_hog = hog_features(X_test)
print(Ftr_hog.shape, Fte_hog.shape)

(10000, 441) (2000, 441)


In [8]:
from sklearn.metrics import accuracy_score  # noqa: E402
from sklearn.svm import LinearSVC  # noqa: E402

svm4 = LinearSVC(C=1.0, max_iter=5000, dual="auto").fit(Ftr_hog, y_train)
print(f"cell=4 accuracy = "
      f"{accuracy_score(y_test, svm4.predict(Fte_hog)):.4f}")

cell=4 accuracy = 0.8660


In [9]:
Ftr_7 = ...  # TODO: hog_features ของ X_train ด้วย cell=7
Fte_7 = ...  # TODO: hog_features ของ X_test ด้วย cell=7
svm7 = LinearSVC(C=1.0, max_iter=5000, dual="auto").fit(Ftr_7, y_train)
print("cell=7 accuracy = ...")  # TODO: accuracy_score(y_test, ...)

TypeError: float() argument must be a string or a real number, not 'ellipsis'

คำตอบข้อ 1: (cell=7 ดีขึ้นหรือแย่ลง เพราะอะไร)

## ข้อ 2 — CNN: เทรนนานขึ้นช่วยแค่ไหน

book.ipynb เทรน CNN 10 epochs (accuracy ~0.87) จงเทรนด้วย **20 epochs** แล้วเทียบ (ใช้ `train_cnn` ด้านล่าง — บน CPU ประมาณ 1 นาที)

In [10]:
import torch  # noqa: E402
import torch.nn as nn  # noqa: E402
from torch.utils.data import DataLoader, TensorDataset  # noqa: E402

Xtr_t = torch.from_numpy(X_train).float().unsqueeze(1) / 255.0
Xte_t = torch.from_numpy(X_test).float().unsqueeze(1) / 255.0
ytr_t = torch.from_numpy(y_train)
mean, std = Xtr_t.mean(), Xtr_t.std()
Xtr_n = (Xtr_t - mean) / std
Xte_n = (Xte_t - mean) / std
print(f"normalize: mean={mean:.4f} std={std:.4f}")


class SmallCNN(nn.Module):
    """CNN จิ๋ว 2 ชั้น conv — ตั้งใจให้เล็กพอรันบน CPU ได้เร็ว."""

    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten(), nn.Linear(32 * 7 * 7, 10),
        )

    def forward(self, x):
        return self.net(x)


def train_cnn(X, y, epochs=10, bs=128, seed=0, log_every=0):
    """เทรน CNN ด้วย Adam คืนโมเดลที่เทรนแล้ว."""
    torch.manual_seed(seed)
    model = SmallCNN()
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss()
    g = torch.Generator().manual_seed(seed)
    dl = DataLoader(
        TensorDataset(X, y), batch_size=bs, shuffle=True, generator=g
    )
    for ep in range(epochs):
        model.train()
        for xb, yb in dl:
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            opt.step()
        if log_every and (ep + 1) % log_every == 0:
            print(f"  epoch {ep + 1}/{epochs}: loss={loss.item():.4f}")
    return model


def cnn_accuracy(model):
    model.eval()
    with torch.no_grad():
        pred = model(Xte_n).argmax(1).numpy()
    return accuracy_score(y_test, pred)

normalize: mean=0.2867 std=0.3535


In [11]:
from sklearn.metrics import accuracy_score  # noqa: E402

EPOCHS = ...  # TODO: 20
cnn20 = train_cnn(Xtr_n, ytr_t, epochs=EPOCHS)
print(f"CNN {EPOCHS} epochs accuracy = ...")  # TODO: cnn_accuracy(cnn20)

TypeError: 'ellipsis' object cannot be interpreted as an integer

คำตอบข้อ 2: (20 epochs ดีขึ้นกว่า 10 กี่จุด — ค่าที่เพิ่มมากไหมเทียบกับเวลาที่จ่ายไป)

## ข้อ 3 — อ่าน confusion matrix: โมเดลสับสนคู่ไหน

ใช้โมเดล CNN จากข้อ 2 แสดง confusion matrix แล้ว**หาคู่คลาส (จริง i, ทำนาย j) ที่สับสนมากที่สุด**นอกแนวทแยง โดยโปรแกรม (เทคนิค: ตั้งแนวทแยงเป็น 0 แล้ว `argmax`)

In [12]:
import matplotlib.pyplot as plt  # noqa: E402
from sklearn.metrics import ConfusionMatrixDisplay  # noqa: E402

cnn20.eval()
with torch.no_grad():
    pred20 = cnn20(Xte_n).argmax(1).numpy()
ConfusionMatrixDisplay.from_predictions(
    y_test, pred20, colorbar=False, xticks_rotation=90
)
plt.show()

cm = np.zeros((10, 10), dtype=int)
for true, pr in zip(y_test, pred20):
    cm[true, pr] += 1
np.fill_diagonal(cm, 0)
worst_i, worst_j = ...  # TODO: np.unravel_index(np.argmax(cm), cm.shape)
print(
    f"สับสนมากสุด: จริง={CLASS_EN[worst_i]} "
    f"ถูกทำนายเป็น={CLASS_EN[worst_j]} ({cm[worst_i, worst_j]} ครั้ง)"
)

NameError: name 'cnn20' is not defined

คำตอบข้อ 3: (คู่สับสนมากสุด — สมเหตุสมผลไหมจากหน้าตาเสื้อผ้าสองคลาสนั้น)

## ข้อ 4 — Learning curve: เติมจุด 50%

book.ipynb เทรน curve ที่ 5% / 25% / 100% จงเติมจุด **50% (5,000 ภาพ)** ด้วยวิธี HOG+SVM (cell=4) แล้วตอบว่าค่าที่ได้อยู่ระหว่างจุด 25% กับ 100% จริงไหม

In [13]:
svm4 = LinearSVC(C=1.0, max_iter=5000, dual="auto").fit(Ftr_hog, y_train)
acc_100 = accuracy_score(y_test, svm4.predict(Fte_hog))
print(f"n=10000: accuracy={acc_100:.4f}")
accs = {500: None, 2500: None, 5000: None}
for n in accs:
    svm_n = ...  # TODO: LinearSVC เหมือนบรรทัดบน แต่ fit(Ftr_hog[:n],
    #             y_train[:n])
    accs[n] = accuracy_score(y_test, svm_n.predict(Fte_hog))
    print(f"n={n}: accuracy={accs[n]:.4f}")

n=10000: accuracy=0.8660


AttributeError: 'ellipsis' object has no attribute 'predict'

คำตอบข้อ 4: (ค่าที่ 50% อยู่ตำแหน่งไหนในช่วง — สอดคล้องกับแนวโน้มข้อมูลน้อย → แย่ลงหรือไม่)

## ข้อ 5 — ตัดสินใจแบบนักเมโทรโลยี (ตอบใน markdown)

สมมติคุณมี **ภาพชิ้นงาน 200 ภาพ** (2 คลาส: ปกติ/ชำรุด) และต้องส่งโมเดลให้หัวหน้าในสัปดาห์นี้ — เลือกวิธีจากทั้ง 3 (HOG+SVM / CNN จากศูนย์ / transfer learning) พร้อมเหตุผล 2 ข้อ อ้างอิง learning curve ที่เห็น